### Adapt from Zaki's implementation

In [ ]:
import pandas as pd
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigs
from scipy.sparse import coo_matrix, csr_matrix
import scipy as sp
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)
import seaborn as sns
import matplotlib as mpl
#set the matplotlib font to arial
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

### Data querying

In [3]:
def load_sqlite_database(sql_path):
    """
    Load metadata and connectivity data from SQLite database.
    
    Parameters:
    -----------
    sql_path : str
        Path to SQLite database
        
    Returns:
    --------
    tuple
        (metadata_df, edgelist_df)
    """
    conn = sqlite3.connect(sql_path)
    
    # Load all tables
    meta_df = pd.read_sql_query("SELECT * FROM meta", conn)
    try:
        edgelist_df = pd.read_sql_query("SELECT * FROM edgelist_simple", conn)
    except Exception as e:
        print("Failed to load 'edgelist_simple'. Attempting to load 'edgelist' instead.")
        edgelist_df = pd.read_sql_query("SELECT * FROM edgelist", conn)
    conn.close()
    return meta_df, edgelist_df

In [4]:
sql_path = '/n/data1/hms/neurobio/wilson/banc/connectivity/manc_1.2.1_data.sqlite'
meta_df, edgelist_df = load_sqlite_database(sql_path)

In [5]:
edgelist_df.columns

Index(['post', 'pre', 'count', 'norm', 'post_count', 'pre_count',
       'post_top_nt', 'post_top_nt_p', 'pre_top_nt', 'pre_top_nt_p'],
      dtype='object')

In [13]:
def filter_edgelist_and_update_meta(
    edgelist_df, 
    meta_df, 
    synaptic_threshold=25, 
    filter_neurotransmitter=False, 
    neurotransmitter_list=None
):
    """
    Filter edgelist DataFrame based on synaptic count threshold and optionally neurotransmitter criteria.
    Update meta_df to remove neurons without any connections.
    
    Parameters:
    -----------
    edgelist_df : DataFrame
        DataFrame containing the edgelist with columns ['pre', 'post', 'count', 'pre_top_nt', ...].
    meta_df : DataFrame
        Metadata DataFrame containing neuron information.
    synaptic_threshold : int, optional
        Minimum total synaptic count to retain a connection. Default is 25.
    filter_neurotransmitter : bool, optional
        Whether to apply neurotransmitter-based filtering. Default is False.
    neurotransmitter_list : list of str, optional
        List of neurotransmitters to filter by (e.g., ['gaba', 'acetylcholine', 'glutamate']).
        Ignored if `filter_neurotransmitter` is False.

    Returns:
    --------
    tuple
        (filtered_edgelist, updated_meta, filtered_out_neurons)
    """
    # Filter by neurotransmitter if the flag is set
    if filter_neurotransmitter:
        if neurotransmitter_list is None:
            neurotransmitter_list = ['gaba', 'acetylcholine', 'glutamate']  # Default neurotransmitters
        edgelist_df = edgelist_df[
            (edgelist_df['pre_top_nt'].isin(neurotransmitter_list))
        ]

    # Group by 'pre' and 'post' and sum synaptic counts
    grouped_edgelist = edgelist_df.groupby(['pre', 'post'], as_index=False).agg(synaptic_count=('count', 'sum'))
    filtered_pairs = grouped_edgelist[grouped_edgelist['synaptic_count'] > synaptic_threshold][['pre', 'post']]

    # Retain only rows in the original edgelist that match the filtered pairs
    filtered_edgelist = edgelist_df.merge(filtered_pairs, on=['pre', 'post'], how='inner')

    # Determine neurons remaining in the filtered edgelist
    remaining_neurons = set(filtered_edgelist['pre']).union(set(filtered_edgelist['post']))
    all_neurons = set(edgelist_df['pre']).union(set(edgelist_df['post']))
    filtered_out_neurons = all_neurons - remaining_neurons

    # Update the meta DataFrame to remove neurons without connections
    updated_meta = meta_df[~meta_df['bodyid'].isin(filtered_out_neurons)].copy()

    return filtered_edgelist, updated_meta, filtered_out_neurons


In [14]:
fw_elistf, fw_meta, filtered_n = filter_edgelist_and_update_meta(edgelist_df, meta_df, 25, True)

In [15]:
fw_meta.columns

Index(['bodyid', 'neuromore', 'total_outputs', 'total_inputs', 'axon_outputs',
       'dend_outputs', 'axon_inputs', 'dend_inputs', 'pd_outputs',
       'pn_outputs', 'pd_inputs', 'pn_inputs', 'total_outputs_density',
       'total_inputs_density', 'axon_outputs_density', 'dend_outputs_density',
       'axon_inputs_density', 'dend_inputs_density', 'total_length',
       'axon_length', 'dend_length', 'pd_length', 'pn_length', 'axon_width',
       'dend_width', 'pd_width', 'pn_width', 'segregation_index',
       'projection_score', 'nodes', 'cable_length', 'dataset', 'post', 'pre',
       'voxels', 'top_nt', 'top_nt_p', 'side', 'nerve', 'hemilineage',
       'cell_class', 'cell_sub_class', 'cell_type', 'type', 'name',
       'other_names', 'origin', 'modality', 'receptor_type', 'soma_location',
       'tosoma_location', 'root_location', 'soma', 'top_p', 'supervoxel_id',
       'known_nt', 'known_nt_source'],
      dtype='object')

In [16]:
fw_elistf.columns

Index(['post', 'pre', 'count', 'norm', 'post_count', 'pre_count',
       'post_top_nt', 'post_top_nt_p', 'pre_top_nt', 'pre_top_nt_p'],
      dtype='object')

In [ ]:
def process_filtered_edgelist_by_nt(filtered_edgelist, invert_nts=None):
    """
    Process the filtered edgelist by adding a new column 'effective_count',
    which adjusts the 'count' values based on specific neurotransmitters in
    the 'pre_top_nt' column.

    Parameters:
    -----------
    filtered_edgelist : DataFrame
        The edgelist DataFrame with columns including 'pre_top_nt' and 'count'.
    invert_nts : list of str, optional
        List of neurotransmitters for which to invert the 'count' sign.
        Default is ['gaba', 'glutamate'].

    Returns:
    --------
    processed_edgelist : DataFrame
        The modified edgelist with a new column 'effective_count'.
    """
    if invert_nts is None:
        invert_nts = ['gaba', 'glutamate']  # Default neurotransmitters to invert

    # Initialize 'effective_count' as a copy of 'count'
    filtered_edgelist['effective_count'] = filtered_edgelist['count']

    # Process each neurotransmitter in the invert list
    for nt in invert_nts:
        filtered_edgelist.loc[
            filtered_edgelist.pre_top_nt.str.contains(nt, case=False, na=False), 
            "effective_count"
        ] *= -1

    return filtered_edgelist


In [18]:
fw_elistf = process_filtered_edgelist_by_nt(fw_elistf)

In [32]:

# Function to convert cell IDs to matrix indices
def convert_ids_to_indices(edge_list):
    """
    Converts `pre` and `post` IDs in the edge list to matrix indices.
    
    Args:
        edge_list: DataFrame with 'pre' and 'post' columns representing connections.
    
    Returns:
        edge_list: Modified DataFrame with 'pre_mat' and 'post_mat' columns.
        n_neurons: Total number of unique neurons (matrix size).
    """
    edge_list.pre = edge_list.pre.astype(int)
    edge_list.post = edge_list.post.astype(int)

    unique_ids_pre = edge_list.pre.unique()
    unique_ids_post = edge_list.post.unique()

    temp_pre = np.full(len(edge_list.pre.values), np.nan)
    temp_post = np.full(len(edge_list.post.values), np.nan)

    for ii, uid in enumerate(unique_ids_pre):
        temp_pre[np.where(edge_list.pre.values == uid)] = ii
        temp_post[np.where(edge_list.post.values == uid)] = ii

    incr = 0
    for ii in range(len(temp_post)):
        if np.isnan(temp_post[ii]):
            temp_post[edge_list.post.values == edge_list.post.values[ii]] = len(unique_ids_pre) + incr
            incr += 1

    edge_list.insert(1, "post_mat", temp_post.astype(int), True)
    edge_list.insert(3, "pre_mat", temp_pre.astype(int), True)

    n_neurons = int(np.max(temp_post)) + 1
    return edge_list, n_neurons

# Function to build a connectivity matrix
def create_connectivity_matrix(edge_list, n_neurons):
    """
    Creates a sparse connectivity matrix from the edge list.
    
    Args:
        edge_list: DataFrame with 'pre_mat', 'post_mat', and 'weight' columns.
        n_neurons: Total number of unique neurons (matrix size).
    
    Returns:
        connectivity_matrix: Sparse CSR matrix.
    """
    rows = edge_list.pre_mat.values
    cols = edge_list.post_mat.values
    weights = edge_list.weight.values  # Assuming a 'weight' column exists

    connectivity_matrix = coo_matrix((weights, (rows, cols)), shape=(n_neurons, n_neurons))
    return csr_matrix(connectivity_matrix)

# Function to perform eigendecomposition
def perform_eigendecomposition(connectivity_matrix, k_eig_vecs=1000):
    """
    Performs eigendecomposition on the connectivity matrix.
    
    Args:
        connectivity_matrix: Sparse connectivity matrix.
        k_eig_vecs: Number of eigenvalues and eigenvectors to compute.
    
    Returns:
        eig_val: Sorted eigenvalues.
        eig_vec_r: Sorted right eigenvectors.
        eig_vec_l: Sorted left eigenvectors.
    """
    eig_val, eig_vec_r = eigs(connectivity_matrix, k=k_eig_vecs)
    _, eig_vec_l = eigs(connectivity_matrix.T, k=k_eig_vecs)

    abs_eig_val = np.abs(eig_val)
    sort_ind = np.argsort(abs_eig_val)[::-1]

    eig_val = eig_val[sort_ind]
    eig_vec_r = eig_vec_r[:, sort_ind]
    eig_vec_l = eig_vec_l[:, sort_ind]
    return eig_val, eig_vec_r, eig_vec_l

# Function to scale the connectivity matrix for stability
def scale_matrix(connectivity_matrix, eig_val):
    """
    Scales the connectivity matrix to ensure stability (largest eigenvalue just below 1).
    
    Args:
        connectivity_matrix: Sparse connectivity matrix.
        eig_val: Largest eigenvalue.
    
    Returns:
        W_scaled: Scaled connectivity matrix.
        scale_factor: Factor used for scaling.
    """
    scale_factor = 0.99 / np.abs(eig_val[0])
    W_scaled = connectivity_matrix * scale_factor
    return W_scaled, scale_factor

# Function to define a seed vector
def define_seed_vector(edge_list, meta_data, n_neurons, target_class='ascending'):
    """
    Defines a seed vector based on specific neuron classes.
    
    Args:
        edge_list: DataFrame with 'pre_mat' and 'post_mat' columns.
        meta_data: DataFrame containing neuron metadata with 'cell_class' and 'bodyid'.
        n_neurons: Total number of unique neurons (vector size).
        target_class: Class of neurons to initialize the seed vector.
    
    Returns:
        seed_vector: Initialized seed vector.
    """
    seed_meta = meta_data[meta_data.cell_class.str.fullmatch(target_class).astype(bool)]
    seed_vector = np.zeros(n_neurons)

    for _, neuron in seed_meta.iterrows():
        bodyid = neuron.bodyid
        ind_post = np.where(edge_list.post == bodyid)
        ind_pre = np.where(edge_list.pre == bodyid)

        if len(ind_post[0]) > 0:
            seed_vector[edge_list.post_mat[ind_post[0][0]]] = 1
        elif len(ind_pre[0]) > 0:
            seed_vector[edge_list.pre_mat[ind_pre[0][0]]] = 1

    return seed_vector


In [29]:
fw_elistf, n_neurons = convert_ids_to_indices(fw_elistf)

In [35]:
seed_v = define_seed_vector(fw_elistf, fw_meta, n_neurons, target_class="ascending")


In [ ]:
def initialize_simulation(seed_v, eig_vec_r, eig_val, scale_orig, n_neurons, T, tau):
    """Initialize the simulation for signal propagation."""
    r = np.full((n_neurons, T), np.nan)
    y = np.full((eig_vec_r.shape[0], T), np.nan)
    r[:, 0] = seed_v
    y[:, 0] = eig_vec_r.T @ seed_v
    return r, y

def solve_dynamics(y, eig_val, scale_orig, T, tau):
    """Solve the dynamics in eigenspace."""
    for tt in range(1, T):
        y[:, tt] = y[:, 0] * np.exp((scale_orig * eig_val) * tt * tau)
    return y

def map_to_neural_space(y, eig_vec_l):
    """Map activity from eigenspace back to neural space."""
    return eig_vec_l @ y

def plot_neural_activity(r, T, filename):
    """Plot neural activity over time."""
    plt.figure()
    for jj in range(100):
        plt.plot(np.arange(0, T), r[jj, :])
    plt.ylim(-10, 10)
    plt.savefig(filename)

def compute_time_to_threshold(r, th):
    """Compute the time to reach a threshold for each neuron."""
    r_positive = np.abs(r)
    tth = np.full(r_positive.shape[0], float('inf'))
    for ii in range(r_positive.shape[0]):
        tth_temp = np.where(r_positive[ii, :] >= th, np.arange(r_positive.shape[1]), float('inf'))
        if np.min(tth_temp) < float('inf'):
            tth[ii] = np.min(tth_temp)
    return tth


def create_distance_dataframe(seed_v, tth, meta_df, elist_df, n_neurons):
    """Create a dataframe for distance to the active neurons."""
    distance_AN = pd.DataFrame({'Cell ID': np.arange(len(seed_v)), 'Distance to AN': tth, 'Class': 'other'})
    annotate_neurons(distance_AN, meta_df, elist_df, n_neurons, 'ascending', 'AN')
    annotate_neurons(distance_AN, meta_df, elist_df, n_neurons, 'descending', 'DN')
    annotate_neurons(distance_AN, meta_df, elist_df, n_neurons, 'convergent', 'cnv')
    return distance_AN

def annotate_neurons(distance_df, meta_df, elist_df, n_neurons, neuron_class, prefix):
    """Annotate neurons based on their class (ascending, descending, convergent)."""
    class_meta = meta_df[meta_df.cell_class.str.fullmatch(neuron_class)]
    class_v = np.full(n_neurons, np.nan)
    for body_id in class_meta.bodyid.values.astype(int):
        ind_post = np.where(elist_df.post == body_id)
        ind_pre = np.where(elist_df.pre == body_id)
        if len(ind_post[0]) > 0:
            class_v[elist_df.post_mat[ind_post[0][0]]] = 1
        elif len(ind_pre[0]) > 0:
            class_v[elist_df.pre_mat[ind_pre[0][0]]] = 1
    distance_df.loc[np.where(class_v * distance_df['Distance to AN'] >= 0)[0], 'Class'] = neuron_class

def plot_distance_scatter(distance_AN, filename="distance_scatter.png"):
    """Plot a scatterplot of distances to the active neurons."""
    fig, ax = plt.subplots(figsize=(5, 4))
    order = distance_AN['Class'].unique()
    sns.stripplot(data=distance_AN, y='Class', x='Distance to AN', alpha=0.1, order=order, ax=ax)
    x = distance_AN.groupby('Class')['Distance to AN'].mean().loc[order]
    y = np.arange(0, len(order))
    ax.scatter(x, y, marker='|', c='k', zorder=100)
    ax.set_xlabel('Distance to AN')
    ax.set_ylabel('')
    ax.grid(True, axis='x', which='both')
    sns.despine(trim=True)
    plt.tight_layout()
    plt.savefig(filename)

# Example of how to use these functions
# r, y = initialize_simulation(seed_v, eig_vec_r, eig_val, scale_orig, n_neurons, T, tau)
# y = solve_dynamics(y, eig_val, scale_orig, T, tau)
# r[:, 1:T] = map_to_neural_space(y, eig_vec_l)
# plot_neural_activity(r, T)
# tth = compute_time_to_threshold(r, th)
# distance_AN = create_distance_dataframe(seed_v, tth, manc_meta, manc_elist, n_neurons)
# plot_distance_scatter(distance_AN)


In [ ]:
# Simulation of signal propagation with Linear dynamics model
T = 300 # Total simulation time
tau = 0.05 # Time constant
r = np.full((n_neurons, T), np.nan)
r[:,0] = seed_v
y = np.full((k_eig_vecs, T), np.nan)
y[:,0] = eig_vec_r.T @ seed_v # Project seed onto eigenvectors using right eigenvectors

# Solve the dynamical equation in eigenspace
for tt in range(1,T):
        y[:,tt] = y[:,0]*np.exp((scale_orig*eig_val)*tt*tau) 
        
r[:,1:T] = eig_vec_l @ y[:,1:T] # Map activity from eigenspace back to neural space using left eigenvectors

plt.figure()
for jj in range(100):
    plt.plot(np.arange(0,T), r[jj,:])
    
plt.ylim(-10, 10)

plt.savefig("AN_signal_propagation_1000_eig_vec.png")
# Measure distance from seed
r_positive = np.abs(r)
th = 1
tth = np.full(len(seed_v), float('inf')) #Time to threshold 
for ii in range(len(seed_v)):
    tth_temp = np.where(r_positive[ii,:]>=th, r_positive[ii,:], float('inf'))
    if min(tth_temp)<float('inf'):
        tth[ii] = np.argmin(tth_temp)

# Construct a new dataframe for distance from seed
distance_AN = pd.DataFrame([[None, None, None]] * len(seed_v), index=np.arange(len(seed_v)),
                             columns=['Cell ID', 'Distance to AN', 'Class'])
distance_AN['Cell ID'] = np.arange(len(seed_v))
distance_AN['Class'] = 'other'
tth_no_inf = tth
tth_no_inf[np.where(tth_no_inf==float('inf'))] = np.nan
distance_AN['Distance to AN'] = tth_no_inf

# Ascending neurons
AN_meta = manc_meta[manc_meta.cell_class.str.fullmatch('ascending').astype(bool)]
AN_v = np.full(n_neurons, np.nan)
ind_1_post = []
ind_1_pre = []
for ii in range(len(AN_meta)):
    ind_1_post = np.where(manc_elist.post == AN_meta.bodyid.values.astype(int)[ii])
    ind_1_pre = np.where(manc_elist.pre == AN_meta.bodyid.values.astype(int)[ii])
    if len(ind_1_post[0])>0:
        AN_v[manc_elist.post_mat[ind_1_post[0][0]]] = 1
    elif len(ind_1_pre[0])>0:
        AN_v[manc_elist.pre_mat[ind_1_pre[0][0]]] = 1


distance_AN.loc[np.where(AN_v*tth_no_inf>=0)[0], 'Class'] = 'ascending'

# Descending neurons
DN_meta = manc_meta[manc_meta.cell_class.str.fullmatch('descending').astype(bool)]
DN_v = np.full(n_neurons, np.nan)
ind_1_post = []
ind_1_pre = []
for ii in range(len(DN_meta)):
    ind_1_post = np.where(manc_elist.post == DN_meta.bodyid.values.astype(int)[ii])
    ind_1_pre = np.where(manc_elist.pre == DN_meta.bodyid.values.astype(int)[ii])
    if len(ind_1_post[0])>0:
        DN_v[manc_elist.post_mat[ind_1_post[0][0]]] = 1
    elif len(ind_1_pre[0])>0:
        DN_v[manc_elist.pre_mat[ind_1_pre[0][0]]] = 1
        
distance_AN.loc[np.where(DN_v*tth_no_inf>=0)[0], 'Class'] = 'descending'

# Convergent neurons
cnv_meta = manc_meta[manc_meta.cell_class.str.fullmatch('convergent').astype(bool)]
cnv_v = np.full(n_neurons, np.nan)
ind_1_post = []
ind_1_pre = []
for ii in range(len(cnv_meta)):
    ind_1_post = np.where(manc_elist.post == cnv_meta.bodyid.values.astype(int)[ii])
    ind_1_pre = np.where(manc_elist.pre == cnv_meta.bodyid.values.astype(int)[ii])
    if len(ind_1_post[0])>0:
        cnv_v[manc_elist.post_mat[ind_1_post[0][0]]] = 1
    elif len(ind_1_pre[0])>0:
        cnv_v[manc_elist.pre_mat[ind_1_pre[0][0]]] = 1
        
distance_AN.loc[np.where(cnv_v*tth_no_inf>=0)[0], 'Class'] = 'convergent'


fig, ax = plt.subplots(figsize=(5, 4))

order = distance_AN.Class.unique()

ax = sns.stripplot(data=distance_AN, y='Class', x='Distance from AN', alpha=.1, order=order, ax=ax)

x = distance_AN.groupby('Class')['Distance to AN'].mean().loc[order]
y = np.arange(0, len(order))

ax.scatter(x, y, marker='|', c='k', zorder=100)

ax.set_xlabel('Distance to AN')
ax.set_ylabel('')

ax.grid(True, axis='x', which='both')

sns.despine(trim=True)

plt.tight_layout()

plt.savefig("AN_to_subclasses_scatter_sig_prop_no_inf.png")